# NB-24b — how many censored utterances are false negatives?

NB-24 asked the right question and was built to answer a harder one than is
answerable. This notebook narrows it to what the data can support.

**What NB-24 established.** On `rung1-final`, 22 of 25 censored utterances
succeeded under a stronger attack — 88%, Wilson 95% [70%, 96%]. A censored
utterance is usually a *failed attack*, not a resistant utterance. That matters
because `reanalysis.py` imputes 679 of 1600 observations (42.4%) at the ceiling
on the opposite assumption.

**Why NB-24 could not finish.** The cost is entirely in the failures. A flip
took about 4 minutes; a non-flip burned all 5 restarts × 2000 steps, about 120
minutes. `rung100-ms3500` has 47 censored rows at roughly a 50% flip rate, so
about 47 GPU-hours. The session was killed by Kaggle's 12 h limit during the
second batch.

**The three changes.**

1. **A capped budget, identical on every checkpoint.** 2 restarts × 500 steps
   instead of 5 × 2000. A non-flip now costs about 12 minutes. This makes a
   flip rate a *lower bound* on the false-negative rate, which is the only
   thing that was ever claimable: no budget proves unreachability. The cap is
   the same everywhere, so rates stay comparable across checkpoints — which is
   why `rung1-final` is re-measured here rather than reusing NB-24's 88%.

2. **A sample, not the whole set.** 16 censored utterances per checkpoint,
   drawn with a recorded seed. That gives a rate to about ±12 points, which is
   enough to tell two checkpoints apart. Enumerating all 47 to pin the third
   digit is not worth five sessions.

3. **A wall clock that actually holds.** NB-24 checked its deadline before each
   batch, then started one that ran 113 minutes past it. This estimates the
   next batch from the batches already done and refuses to start one that will
   not fit, the way NB-23 does.

**The question this answers.** Not "what is the false-negative rate" — "does it
differ between low and high exposure". A uniform rate attenuates the slope; a
rate that varies with exposure confounds it. `reanalysis.py` section 6c-4
computes how large the high-exposure rate would have to be to take the slope to
zero, and this notebook measures whether it is that large.

Outputs are written after every batch and the notebook resumes from its own
published output, so a killed session loses nothing.


## 1 · What to re-attack, and the deliberately stronger settings

In [ ]:
NOTEBOOK = "NB-24b-false-negative-rate"

# rung1-final is NOT here. NB-24 already measured it at 22/25 = 88%, and
# because restarts 3 and 4 contributed no flips, that figure IS the 3 x 2000
# result -- re-running it would spend 3 GPU-h to reproduce a number we hold.
# Its CSV is in the repository at data/results/strength_rung1-final_T5.csv.
#
# What is missing is the HIGH-exposure rate, which is what decides whether the
# censoring bias is uniform (slope attenuated, comparisons valid) or
# differential (comparisons confounded). reanalysis.py section 6c-3 shows the
# slope goes to +0.1220 at phi_hi = 0.50 and to -0.0087 only if the LOW end is
# near zero -- which is already excluded, measured at 0.88.
#
# ms3500 first: it is the decisive one, and 4 of its 16 are already done.
QUEUE = [
    dict(tag="rung100-ms3500", ckpt_dataset="ckpt-xlsr-100h", rung=100,
         checkpoint="milestone-3500",
         source_csv="eps50_rung100_milestone-3500_T5_n100.csv"),
    dict(tag="rung25-final", ckpt_dataset="ckpt-xlsr-25h", rung=25,
         checkpoint="final", source_csv="eps50_rung25_final_T5_n100.csv"),
]

WALL_HOURS = 11.0
MIN_HOURS_TO_START = 0.6          # a condition needs this much left to begin
SAFETY = 1.15                     # start a batch only if 1.15x its estimate fits

# --- unchanged, so the comparison is like-for-like -------------------------
EDIT_TYPE  = "T5"
N_EVAL     = 100
BATCH_SIZE = 4            # fp32 needs more memory per sample
EPS_HI     = 2e-3         # ceiling only: this is a yes/no question
SEED       = 1337

# --- the sample, fixed here rather than drawn at runtime -------------------
# 16 of the censored rows per checkpoint. At a true rate of 0.5 the 95%
# interval is about +/-12 points -- more than wide enough to separate 0.88 from
# 0.4, which is the comparison that matters.
#
# The ids are EMBEDDED. NB-24 had to glob three published result datasets to
# find out which utterances were censored, which is one more way a session can
# die after the model is already loaded -- the failure mode that cost NB-26 a
# run. The draw was made locally from the published CSVs in `data/results/`
# with random.Random(f"{SAMPLE_SEED}-{tag}").sample(sorted(censored), 16), so
# it is reproducible from the repository and auditable here in the source.
#
# If a source CSV happens to be attached the notebook checks the embedded list
# against it, but it does not require one.
SAMPLE_N    = 16
SAMPLE_SEED = 20261112

SAMPLE = {
    # The ms3500 draw keeps the 4 utterances NB-24 already measured and adds 12
    # from the remaining 43, so the GPU time already spent counts toward the
    # rate. Seed "20261112-rung100-ms3500-v2".
    "rung100-ms3500": [
        "01984ba63da8", "243889b3779d", "244778fcaa1b", "335d8208bf04",
        "3f815dd3106e", "49ea01e164e4", "4bdf58bd8e1c", "68ac2254af9d",
        "6f961c5164e4", "761daac20869", "91a15db20e5f", "b3c1cfc26de0",
        "d6903f1df327", "f45c9b5db0d5", "f58039c38c9a", "fde774dcabaf",
    ],
    "rung25-final": [
        "017c67ca09ac", "14bcc5831d17", "18c53133363f", "219228fe3cdd",
        "243889b3779d", "244778fcaa1b", "6f961c5164e4", "82e1ca576ecf",
        "833f9edf0446", "86fe4a267db9", "91b4a00568ea", "9f7f11baf658",
        "b3c1cfc26de0", "d6903f1df327", "ebf7c9d6d671", "ed0e3995715d",
    ],
}

# Already measured by NB-24 at a budget that 3 x 2000 reproduces exactly. If
# the salvaged CSV is attached these are skipped; if it is not, they are simply
# re-run and the result is unchanged.
ALREADY_DONE = {
    "rung100-ms3500": ["01984ba63da8", "244778fcaa1b", "d6903f1df327",
                       "f45c9b5db0d5"],
}

# Censored counts the draws came from, for the record and for the audit below.
CENSORED_TOTAL = {"rung25-final": 34, "rung100-ms3500": 47}

assert set(SAMPLE) == {c["tag"] for c in QUEUE}, "SAMPLE must cover the queue exactly"
for _t, _ids in SAMPLE.items():
    assert len(_ids) == SAMPLE_N, f"{_t}: {len(_ids)} ids, expected {SAMPLE_N}"
    assert len(set(_ids)) == SAMPLE_N, f"{_t}: duplicate ids"
    assert _ids == sorted(_ids), f"{_t}: ids must be sorted, or the audit is unstable"

# --- stronger than published, trimmed to what NB-24 actually needed -------
# NB-24's run was killed, but its per-flip record survived and it says exactly
# where to cut. Of its 22 flips at rung1-final:
#
#   restart 0: 13 flips, at steps 100..950
#   restart 1:  6 flips, at steps 200..1850
#   restart 2:  3 flips, at steps 100..1700
#   restarts 3 and 4: NOTHING
#
# So 3 x 2000 reproduces all 22 of 22 and 5 x 2000 bought nothing, at 60% of
# the cost. Do not cut further. A shallow cap looks affordable and is not
# usable: 2 x 500 would have caught 14 of 22 at rung1-final but ZERO of the 2
# at rung100-ms3500, whose flips needed 1100 and 1500 steps on restart 0. The
# gap it reports would be the budget, not the models -- manufacturing the exact
# conclusion this notebook exists to test.
#
# The flip rate is still a LOWER BOUND on the false-negative rate. No budget can
# prove an utterance unreachable, so a lower bound is the only claim available
# at any cost. What matters is that the bound is identical on every checkpoint.
USE_AMP        = False    # fp32; AMP overflow skips steps for a whole batch
STRONG_STEPS   = 2000     # against 400 probe / 800 confirm published
N_RESTARTS     = 3        # 0 from zero (as published), 1-2 random in the ball
LR_SCALE       = 0.05     # lr = LR_SCALE * eps; at the ceiling exactly 1e-4,
                          # the published LR, so a flip is the extra effort and
                          # not a change of step size
STALL_PATIENCE = 10 ** 9  # off
CHECK_EVERY    = 50       # as NB-24; at 2000 steps this costs at most 2.5%

# These exist only because the cloned core cells read them.
EPS_LO, BISECT_ROUNDS, PROBE_STEPS, CONFIRM_STEPS = 1e-4, 0, STRONG_STEPS, STRONG_STEPS
FIXED_EPS, LR, VALIDATE_BATCHING = EPS_HI, LR_SCALE * EPS_HI, False

assert USE_AMP is False, "the whole point is fp32; AMP overflow skips steps"
assert N_RESTARTS >= 2, "restart 0 reproduces the published start; 1+ is the escalation"
assert STRONG_STEPS >= 2000, (
    "NB-24 recorded flips as deep as 1850 steps, and both rung100-ms3500 flips "
    "needed 1100+. A shallower budget under-counts the high-exposure end, which "
    "is the end this notebook exists to measure.")
assert N_RESTARTS >= 3, (
    "NB-24's restart 2 produced 3 of its 22 flips; restarts 3 and 4 produced "
    "none. Three is the measured floor, five was waste.")
assert abs(LR_SCALE * EPS_HI - 1e-4) < 1e-12, (
    "LR_SCALE must reproduce the published 1e-4 at the ceiling, or a flip could "
    "be the learning rate rather than the extra effort.")
assert SAMPLE_N >= 12, "below ~12 the interval is too wide to separate two rates"
assert CHECK_EVERY * 4 <= STRONG_STEPS, "too few success checks inside the budget"

_budget = N_RESTARTS * STRONG_STEPS
_published = 400
# NB-24: a batch of 4 containing one permanent failure ran the full 5 x 2000 =
# 10000 iterations in about 120 min, so roughly 0.72 s an iteration.
_sec_per_iter = 0.72
_min_per_batch = _budget * _sec_per_iter / 60
print(f"{NOTEBOOK} | {len(QUEUE)} checkpoints | ceiling {EPS_HI:.0e} only")
print(f"  fp32, {N_RESTARTS} restarts x {STRONG_STEPS} steps = {_budget} step-budget")
print(f"  {_budget/_published:.1f}x the published 400-step probe, "
      f"{_budget/(5*2000):.1f}x NB-24's 10000 -- which reproduced all 22 of its flips")
print(f"  {SAMPLE_N} censored utterances per checkpoint")
_todo = sum(SAMPLE_N - len(ALREADY_DONE.get(c["tag"], [])) for c in QUEUE)
print(f"  {_todo} utterances to measure, ~{_min_per_batch:.0f} min per batch of "
      f"{BATCH_SIZE} in the worst case")
print(f"  worst-case total ~{-(-_todo // BATCH_SIZE) * _min_per_batch / 60:.1f} h "
      f"against WALL_HOURS = {WALL_HOURS}")
print()
print("A censored utterance that flips here was a FALSE NEGATIVE in the published")
print("run. One that does not flip is NOT shown to be resistant: this budget is")
print("a lower bound on the false-negative rate, not a proof of resistance.")


## 2 · Boilerplate

In [ ]:
import os, hashlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata, gc
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

# PROV holds only what is defined AT THIS POINT. In NB-05 the rung, edit type
# and bracket were module-level constants, so they could be read here; in a
# queue runner they are per-condition and are not bound yet. Reading them here
# is a NameError that fires three cells before anything expensive starts. The
# run-specific fields are added by the summary cell at the end.
# Project-wide Bengali normaliser. Defined HERE, in boilerplate, because the
# input-resolution cell asserts with it and in NB-05 it lived in the (later)
# target-construction cell -- which a queue runner does not have.
ZW = dict.fromkeys(map(ord, "\u200c\u200d"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

PROV = {"notebook": NOTEBOOK, "seed": SEED,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# Internet is off in this session, so `pip install transformers==4.44.2` cannot
# reach the index. Measured twice in this project's own logs (NB-12 and NB-13,
# 2026-09-30): the cell spends ~202 s timing out, prints two ERROR lines, fails,
# and leaves the preinstalled version in place anyway. Pure waste.
#
# So we do not attempt the pin. We import what is already present and RECORD the
# version instead -- the pin's purpose was reproducibility, and a recorded
# version serves that better than a failed install does.
#
# If internet is ever turned ON, restore the pin: 4.44.2 is the version every
# figure measured before 2026-09-30 was produced on, and mixing versions across
# figures is exactly what the provenance record exists to catch.
#
# `jiwer` is deliberately NOT imported: it is not preinstalled on Kaggle, and
# the only place this notebook would use it is a FIXED_EPS clean-WER branch
# that cannot execute (FIXED_EPS is None here) and already guards itself.
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa, numpy
print(f"transformers {transformers.__version__} | numpy {numpy.__version__} "
      f"| librosa {librosa.__version__}")
if transformers.__version__ != "4.44.2":
    print(f"  NOTE: this is NOT the project pin (4.44.2). Record "
          f"{transformers.__version__} against every number this run produces.")
PROV["transformers"] = transformers.__version__
PROV["numpy"] = numpy.__version__


## 3 · Inputs, eval set and frozen targets

In [ ]:
# ---------------------------------------------------------------------------
# Kaggle does not use one mount layout. Depending on the kernel you get
#     /kaggle/input/<name>                (older kernels)
#     /kaggle/input/datasets/<name>       (your own datasets, current)
#     /kaggle/input/competitions/<name>   (competition data)
# A flat glob only sees the wrapper dirs -- NB-01 failed with exactly this,
# reporting: Visible inputs: ['competitions', 'datasets']
# ---------------------------------------------------------------------------
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, ("Attach bn-asr-splits -- test.csv is needed to resolve audio paths "
                "for the 100 target utterances.\n"
                f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
print("splits:", SPLITS)

# ---------------------------------------------------------------------------
# EVAL SET AND TARGETS: read, never regenerated.
#
# Task B's targets are published inside every eps50-*-n100 dataset as
# task_b_targets_n100.csv. Reading them makes this run byte-comparable by
# construction rather than by reproducing a generator. The hash is checked.
# ---------------------------------------------------------------------------
TGT_CSV = next((f"{p}/task_b_targets_n100.csv" for p in inputs
                if os.path.exists(f"{p}/task_b_targets_n100.csv")), None)
assert TGT_CSV, ("task_b_targets_n100.csv not found. Attach ONE Task B result "
                 "dataset, e.g. zoayriaabedin/eps50-rung1-final-t5-n100.\n"
                 f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
TGT_SHA = hashlib.sha256(open(TGT_CSV, "rb").read()).hexdigest()
print("targets:", TGT_CSV)
print("targets sha256:", TGT_SHA)
assert TGT_SHA == "8711dc56202b33f90e769c9735231b0fa8cf27fcbc20151f5466bcd9cc2b1256", (
    "These are NOT the targets the six Task B runs used. The comparison against "
    "run 1's 75% would be meaningless.")
tg = pd.read_csv(TGT_CSV)
assert len(tg) == 100 and tg.utt_id.nunique() == 100

_full = pd.read_csv(f"{SPLITS}/test.csv")
# source_tokens / target_tokens are carried through: the results writer uses
# both, and eps50_per_token divides by target_tokens. Dropping them here would
# KeyError at save time, after the whole sweep had run.
ev = tg[["utt_id", "target_text", "source_tokens", "target_tokens"]].merge(
        _full, on="utt_id", how="left")
assert ev["path"].notna().all(), "some target utterances are absent from test.csv"
assert (ev.text.map(norm_bn).values == tg.text.map(norm_bn).values).all(), \
    "transcript mismatch between the target file and test.csv"
EV_CSV = TGT_CSV
PROV["attack_eval_csv"] = os.path.basename(TGT_CSV)
PROV["targets_sha256"] = TGT_SHA
PROV["fixed_eps"] = FIXED_EPS
PROV["bisect_rounds"] = BISECT_ROUNDS
# The checkpoint dataset and model_dir are PER CONDITION here, so they are not
# recorded once at the top: every result row carries its own `model_dir`, and the
# queue summary records the tags.
for _c in ("path","text","target_text","source_tokens","target_tokens","duration"):
    assert _c in ev.columns, f"ev is missing {_c}, which a later cell needs"
print(f"eval set: {len(ev)} utterances, manifest order (original 25 first)")

# ---------------------------------------------------------------------------
# Re-resolve the audio root, same hazard as NB-01: NB-00 baked in the mount
# point from ITS session. Here the whole attack set is only 25 files, so check
# every one of them rather than sampling.
# ---------------------------------------------------------------------------
def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0):
        return df, os.path.dirname(p0)
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}",
                           f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, (f"Audio directory '{stem}' not found under /kaggle/input.\n"
                   "Attach: + Add Input -> Competitions -> 'bengaliai speech'.\n"
                   f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    df = df.copy()
    df["path"] = cands[0] + "/" + df["utt_id"].astype(str) + ext
    return df, cands[0]

ev, AUDIO_ROOT = resolve_audio(ev)
absent = [p for p in ev["path"] if not os.path.exists(p)]
assert not absent, f"{len(absent)}/{len(ev)} attack audio files missing, e.g.\n  {absent[0]}"
print(f"audio root: {AUDIO_ROOT}")
print(f"attack set: {len(ev)} utterances, all present")

## 4 · Front end, loss, PGD, bisection

Byte-identical to NB-05. These read `model`, `processor`, `BLANK_ID` and `DO_NORMALIZE` as globals at call time, which is what lets Stage 1 swap checkpoints without redefining anything.

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

In [ ]:
def pgd_batch(x, mask, tgt_pad, tgt_len, targets, eps_vec,
              steps, delta_init=None, lr=LR):
    """eps_vec: (B,) per-sample L-inf budgets.
       Returns (success bool array, frozen deltas, steps_used)."""
    B = x.shape[0]
    eps_col = eps_vec.view(-1, 1)
    delta = (torch.zeros_like(x) if delta_init is None
             else torch.max(torch.min(delta_init.clone(), eps_col), -eps_col))
    delta.requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x)
    success = np.zeros(B, dtype=bool)
    best_loss = torch.full((B,), float("inf"), device=DEVICE)
    stall = torch.zeros(B, dtype=torch.long, device=DEVICE)

    for step in range(steps):
        if not active.any():
            break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)

        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if delta.grad is not None:
            delta.grad[~active] = 0          # finished samples stop moving
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

        with torch.no_grad():                # stall detection
            improved = per < best_loss - 1e-3
            best_loss = torch.minimum(best_loss, per.detach())
            stall = torch.where(improved, torch.zeros_like(stall), stall + 1)
            give_up = active & (stall > STALL_PATIENCE)
            active = active & ~give_up

        if (step + 1) % CHECK_EVERY == 0:
            hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach()
                        success[i] = True
                        active[i] = False
    return success, frozen, step + 1

In [ ]:
def eps50_batch(x, mask, tgt_pad, tgt_len, targets, verbose=True):
    """Per-sample geometric bisection for the smallest budget that still works.

    ---------------------------------------------------------------------------
    BUG FIXED 2026-09-22 (found in the rung-1 T5 run, 40% censoring).

    The previous version ended with

        ok, best, _ = pgd_batch(..., hi, CONFIRM_STEPS, delta_init=best)

    which REASSIGNED `best` wholesale to pgd_batch's return value. That return
    is a `frozen` buffer holding zeros for any sample that did not succeed on
    that particular pass -- so one unlucky final pass wiped out a delta that had
    already been verified at a larger budget. Those samples were then written
    out as censored, with SNR 70-86 dB (the signature of delta == 0) even though
    the ceiling probe had already proved them attackable.

    In the rung-1 run that discarded ~6 of 25 real measurements. Worse, it does
    so at a RATE THAT VARIES BY MODEL, so comparing median eps50 across rungs
    would have been comparing differently-biased subsamples -- straight into
    RQ1's headline.

    Fix: keep a running `best_eps` / `best_delta` that is only ever updated on a
    VERIFIED success. Nothing overwrites a confirmed result. Censoring now means
    exactly one thing: the attack failed at every budget tried, ceiling included.

    Side benefit: eps50 becomes a guaranteed UPPER BOUND on the true minimum.
    Optimiser noise can only make it conservative, never wrong in an
    unpredictable direction -- which is the property you want in a metric you
    are about to compare across four models.
    ---------------------------------------------------------------------------
    """
    B = x.shape[0]
    lo = torch.full((B,), EPS_LO, device=DEVICE)
    hi = torch.full((B,), EPS_HI, device=DEVICE)
    best_eps   = torch.full((B,), float("nan"), device=DEVICE)   # verified only
    best_delta = torch.zeros_like(x)

    def record(okt, eps_vec, d):
        """Adopt a result only when it is both successful and better."""
        nonlocal best_eps, best_delta
        better = okt & (torch.isnan(best_eps) | (eps_vec < best_eps))
        best_eps   = torch.where(better, eps_vec, best_eps)
        best_delta = torch.where(better.view(-1, 1), d, best_delta)

    # Ceiling probe. Failure here is genuine right-censoring.
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, hi, CONFIRM_STEPS)
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, hi, d)
    feasible = ok.copy()
    if verbose:
        print(f"  ceiling probe eps={EPS_HI:.1e}: {feasible.sum()}/{B} feasible, "
              f"{(~feasible).sum()} censored")

    for rnd in range(BISECT_ROUNDS):
        mid = torch.sqrt(lo * hi)
        ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                             PROBE_STEPS, delta_init=best_delta)
        ok = ok & feasible
        okt = torch.tensor(ok, device=DEVICE)
        hi = torch.where(okt, mid, hi)
        lo = torch.where(okt, lo, mid)
        record(okt, mid, d)
        if verbose:
            print(f"  round {rnd+1}/{BISECT_ROUNDS}: {ok.sum()}/{B} succeeded "
                  f"| median eps {mid.median().item():.3e}")

    # Budget-consistency round: one more bisection at the FULL step budget, so
    # the reported boundary is measured at the budget eps50 is defined at rather
    # than at the cheaper probe budget. See RUNLOG for why this matters.
    #
    # Skipped in FIXED_EPS mode: there is no bracket to refine. The ceiling probe
    # already ran at CONFIRM_STEPS, so the single reported budget is measured at
    # the full budget and needs no second pass.
    if BISECT_ROUNDS == 0:
        eps       = best_eps.cpu().numpy()
        confirmed = ~np.isnan(eps)
        if verbose:
            print(f"  -> FIXED_EPS {EPS_HI:.1e}: {confirmed.sum()}/{B} succeeded, "
                  f"{(~confirmed).sum()} resisted")
        return eps, np.ones(B), confirmed, best_delta

    mid = torch.sqrt(lo * hi)
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                         CONFIRM_STEPS, delta_init=best_delta)
    ok = ok & feasible
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, mid, d)
    if verbose:
        print(f"  budget-consistency round @ {CONFIRM_STEPS} steps: {ok.sum()}/{B}")

    eps       = best_eps.cpu().numpy()
    confirmed = ~np.isnan(eps)                 # censored == never succeeded
    width     = (hi / lo).cpu().numpy()
    if verbose:
        print(f"  -> {confirmed.sum()}/{B} measured, {(~confirmed).sum()} censored")
    return eps, width, confirmed, best_delta


## 5 · Per-condition loader

In [ ]:
def load_condition(cfg):
    """Resolve and load one checkpoint, rebinding the globals the attack closures read.

    The closures in the cells above look up `model`, `processor`, `BLANK_ID` and
    `DO_NORMALIZE` at CALL time, not definition time, so rebinding them here is
    all that is needed to point the whole machine at a different checkpoint.
    Every one of the four must be rebound: leaving a stale BLANK_ID or
    DO_NORMALIZE behind would silently attack the new model through the old
    model's decoding assumptions.
    """
    global model, processor, BLANK_ID, DO_NORMALIZE, MODEL_DIR, RUNG, CHECKPOINT
    ds = cfg["ckpt_dataset"]
    cands = [p for p in inputs if re.search(rf"{re.escape(ds)}$", os.path.basename(p))]
    assert len(cands) == 1, (
        f"Need exactly one input matching '{ds}', found {len(cands)}: "
        f"{[os.path.basename(c) for c in cands]}")
    ck = cands[0]
    if cfg["checkpoint"] == "final":
        MODEL_DIR = ck if os.path.exists(f"{ck}/config.json") else f"{ck}/final"
    else:
        MODEL_DIR = f"{ck}/milestones/step-{cfg['checkpoint'].split('-')[1]}"
    assert os.path.exists(f"{MODEL_DIR}/config.json"), f"no config.json under {MODEL_DIR}"

    processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    model.config.apply_spec_augment = False
    DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
    BLANK_ID = model.config.pad_token_id
    RUNG, CHECKPOINT = cfg["rung"], cfg["checkpoint"]
    print(f"  loaded {MODEL_DIR}")
    print(f"  do_normalize {DO_NORMALIZE} | blank {BLANK_ID} | vocab {model.config.vocab_size}")
    return MODEL_DIR


def unload():
    global model, processor
    try:
        del model, processor
    except NameError:
        pass
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 6 · The stronger attack, capped

Restart 0 starts at zero, exactly where the published attack starts, so
anything that flips during restart 0 within 400 steps would have flipped in the
published run too and is a pure nondeterminism artefact. Later restarts begin
at random points inside the eps-ball. `restart_used` and `steps_used` are
recorded for every row, which turns the result into a dose-response curve
rather than a single rate.


In [ ]:
import random

def strong_attack(x, mask, tgt_pad, tgt_len, targets, eps, n_restarts=N_RESTARTS,
                  steps=STRONG_STEPS):
    """Attack one batch at a single budget, with restarts. Returns per-sample
    (success, restart_index_that_worked, steps_used)."""
    B = x.shape[0]
    eps_col = torch.full((B, 1), eps, device=DEVICE)
    lr = LR_SCALE * eps

    success = np.zeros(B, dtype=bool)
    which = np.full(B, -1, dtype=int)
    used = np.zeros(B, dtype=int)
    best_delta = torch.zeros_like(x)

    for rs in range(n_restarts):
        todo = ~success
        if not todo.any():
            break
        # restart 0 starts at zero, matching the published attack; the rest
        # start uniformly inside the eps-ball.
        if rs == 0:
            delta = torch.zeros_like(x)
        else:
            delta = (torch.rand_like(x) * 2 - 1) * eps
        delta = delta.clone().requires_grad_(True)
        opt = torch.optim.Adam([delta], lr=lr)
        active = torch.tensor(todo, device=DEVICE)

        for step in range(steps):
            if not active.any():
                break
            x_adv = torch.clamp(x + delta, -1, 1) * mask
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)   # fp32, no autocast
            loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)
            opt.zero_grad(); loss.backward()
            if delta.grad is not None:
                delta.grad[~active] = 0
            opt.step()
            with torch.no_grad():
                delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

            if (step + 1) % CHECK_EVERY == 0:
                hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
                with torch.no_grad():
                    for i in range(B):
                        if active[i] and hyp[i] == targets[i]:
                            success[i] = True; which[i] = rs; used[i] = step + 1
                            best_delta[i] = delta[i].detach()
                            active[i] = False
        for i in range(B):
            if not success[i]:
                used[i] += steps
    return success, which, used, best_delta

## 7 · Run it on the censored utterances only

In [ ]:
import random as _rnd

def pick_sample(cfg):
    """The embedded sample, cross-checked against the published CSV when that
    file happens to be attached. The check is advisory: a missing source file
    must not stop a run, because the ids are already fixed."""
    chosen = list(SAMPLE[cfg["tag"]])
    n_cen = CENSORED_TOTAL[cfg["tag"]]
    print(f"  {len(chosen)} of {n_cen} censored utterances, embedded sample "
          f"(seed {SAMPLE_SEED}-{cfg['tag']})")

    hits = []
    for d in range(1, 5):
        hits += sorted(glob.glob("/kaggle/input/" + "*/" * d + cfg["source_csv"]))
    if not hits:
        print(f"  {cfg['source_csv']} not attached; skipping the audit "
              "(the sample does not depend on it)")
        return chosen, n_cen
    df = pd.read_csv(hits[0])
    cen = set(df[df.censored.astype(str).str.lower().isin(["true", "1"])].utt_id)
    stray = [u for u in chosen if u not in cen]
    assert not stray, (
        f"{stray} are in the embedded sample but are NOT censored in "
        f"{hits[0]}. The sample was drawn from a different version of this "
        "result file; stop and redraw it rather than measuring the wrong rows.")
    if len(cen) != n_cen:
        print(f"  NOTE: {hits[0]} has {len(cen)} censored rows, the sample was "
              f"drawn from {n_cen}. The {len(chosen)} ids are still valid "
              "censored rows, but the denominator for a whole-set rate changed.")
    print(f"  audit passed against {hits[0]}: all {len(chosen)} are censored there")
    return chosen, len(cen)


def _as_bool(v, default=False):
    """Read a boolean that has been through a CSV. Never use astype(bool) for
    this: the STRING "False" is truthy, so it would silently invert a result."""
    if isinstance(v, bool):
        return v
    if v is None:
        return default
    s = str(v).strip().lower()
    if s in ("true", "1", "yes"):
        return True
    if s in ("false", "0", "no"):
        return False
    if s in ("", "nan", "none"):
        return default
    raise ValueError(f"cannot read {v!r} as a boolean")


def normalise_resumed(rec):
    """Bring a row from an older run up to this notebook's schema.

    NB-24 wrote 13 columns; this notebook writes 18. Leaving the 5 missing ones
    absent makes pandas fill NaN, and `int(df.within_published_budget.sum())`
    in the summary cell then raises on an object column containing NaN --- at
    the very end, after every GPU-hour has been spent. So fill them here.

    `within_published_budget` is RECOVERABLE rather than guessed: the old rows
    carry restart_used and steps_used, which is all it is computed from.
    `n_restarts` and `strong_steps` record what the OLD run actually used, not
    this one's, so the provenance of a resumed row stays truthful.
    """
    r = dict(rec)
    r["now_success"] = _as_bool(r.get("now_success"))
    r["was_censored"] = _as_bool(r.get("was_censored"), True)
    if "within_published_budget" not in r or str(r.get("within_published_budget")).strip().lower() in ("", "nan", "none"):
        r["within_published_budget"] = bool(
            r["now_success"] and int(r.get("restart_used", -1)) == 0
            and int(r.get("steps_used", 10 ** 9)) <= 400)
    else:
        r["within_published_budget"] = _as_bool(r["within_published_budget"])
    # NB-24's budget, which is what these rows were measured at. 3 x 2000
    # reproduces every flip it found, which is why they are reusable at all.
    r.setdefault("n_restarts", 5)
    r.setdefault("strong_steps", 2000)
    r.setdefault("sampled_from", CENSORED_TOTAL.get(r.get("tag"), None))
    r.setdefault("sample_seed", "NB-24 (pre-sampling run)")
    return r


def resume_rows(cfg):
    """Rows this condition already finished, from a previously published run of
    this notebook OR from NB-24. Matched on filename, so attaching the NB-24
    notebook itself works: a notebook's output mounts at
    /kaggle/input/notebooks/<owner>/<slug>/, which the d=3 glob reaches.

    Only rows in this condition's sample are kept. NB-24 measured utterances
    chosen by eval order, not by our draw, so some of its rows are outside the
    sample and counting them would change the denominator.
    """
    name = f"strength_{cfg['tag']}_{EDIT_TYPE}.csv"
    want = set(SAMPLE[cfg["tag"]])
    hits = []
    for d in range(1, 5):
        hits += sorted(glob.glob("/kaggle/input/" + "*/" * d + name))
    for h in hits:
        try:
            df = pd.read_csv(h)
        except Exception as e:
            print(f"  could not read {h}: {e}")
            continue
        if not len(df):
            continue
        recs = [normalise_resumed(r) for r in df.to_dict("records")]
        keep = [r for r in recs if r["utt_id"] in want]
        drop = len(recs) - len(keep)
        print(f"  RESUMING from {h}: {len(recs)} rows, {len(keep)} in this "
              f"condition's sample" + (f", {drop} outside it ignored" if drop else ""))
        if keep:
            return keep
    print(f"  no resume source for {name} -- starting from scratch")
    return []


SESSION_START = time.time()
DEADLINE = SESSION_START + WALL_HOURS * 3600
summary, rows_all = [], []

for qi, cfg in enumerate(QUEUE, 1):
    left = (DEADLINE - time.time()) / 3600
    print("\n" + "=" * 74)
    print(f"[{qi}/{len(QUEUE)}] {cfg['tag']}   ({left:.1f} h left)")
    print("=" * 74)
    if left < MIN_HOURS_TO_START:
        print(f"  only {left:.2f} h left, under MIN_HOURS_TO_START="
              f"{MIN_HOURS_TO_START}: skipping so the output still commits.")
        continue

    chosen, n_cen = pick_sample(cfg)

    rows = resume_rows(cfg)
    done = {r["utt_id"] for r in rows}
    _pre = [u for u in ALREADY_DONE.get(cfg["tag"], []) if u not in done]
    if _pre:
        print(f"  NOTE: {len(_pre)} of this condition's sample were measured by "
              f"NB-24 at a budget 3 x 2000 reproduces,")
        print(f"        but that CSV is not attached, so they will be re-run: "
              f"{_pre}")
        print(f"        Attach the salvaged NB-24 output to skip them.")
    todo = [u for u in chosen if u not in done]
    sub_all = ev[ev.utt_id.isin(todo)].reset_index(drop=True)
    print(f"  {len(todo)} of {len(chosen)} sampled utterances still to do")
    if len(sub_all) == 0:
        print("  this condition is already complete.")
    else:
        load_condition(cfg)
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    OUT = f"{WORK}/strength_{cfg['tag']}_{EDIT_TYPE}.csv"
    if rows:
        pd.DataFrame(rows).to_csv(OUT, index=False)   # keep resumed rows safe

    batches = [sub_all.iloc[i:i+BATCH_SIZE] for i in range(0, len(sub_all), BATCH_SIZE)]
    bt = []                                            # seconds per batch so far
    for bi, sub in enumerate(batches):
        left_s = DEADLINE - time.time()
        if bt:
            est = sum(bt) / len(bt)
            if est * SAFETY > left_s:
                print(f"  *** stopping: next batch ~{est/60:.0f} min, only "
                      f"{left_s/60:.0f} min left. {len(batches)-bi} batches "
                      f"unmeasured in this condition. ***")
                break
        elif left_s < 0:
            print("  *** wall clock already reached ***")
            break

        xb, mb = load_batch(sub)
        tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE)
              for t in sub.target_text]
        tl = torch.tensor([len(t) for t in tp], device=DEVICE)
        tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
        t0 = time.time()
        ok, which, used, best = strong_attack(xb, mb, tpad, tl,
                                              list(sub.target_text), EPS_HI)
        dt = time.time() - t0
        bt.append(dt)
        x_adv = int16_rt(torch.clamp(xb + best, -1, 1)) * mb
        snrs = snr_db_b(xb, x_adv, mb)
        hyp = decode_b(x_adv, mb)
        for i, (_, r) in enumerate(sub.iterrows()):
            # A flip during restart 0 inside the published 400 steps would have
            # happened in the published run as well: that row is pure
            # nondeterminism, not a budget effect.
            within_published = bool(ok[i]) and int(which[i]) == 0 and int(used[i]) <= 400
            rows.append(dict(utt_id=r["utt_id"], tag=cfg["tag"], rung=cfg["rung"],
                             checkpoint=cfg["checkpoint"], duration=r["duration"],
                             was_censored=True, now_success=bool(ok[i]),
                             restart_used=int(which[i]), steps_used=int(used[i]),
                             within_published_budget=within_published,
                             n_restarts=N_RESTARTS, strong_steps=STRONG_STEPS,
                             eps=EPS_HI, sampled_from=n_cen,
                             sample_seed=f"{SAMPLE_SEED}-{cfg['tag']}",
                             snr_db=float(snrs[i]) if ok[i] else None,
                             decoded=(hyp[i] if ok[i] else None),
                             model_dir=MODEL_DIR))
        pd.DataFrame(rows).to_csv(OUT, index=False)
        nleft = len(batches) - bi - 1
        est = sum(bt) / len(bt)
        print(f"    batch {bi+1}/{len(batches)}  {dt/60:5.1f} min  "
              f"{int(ok.sum())}/{len(sub)} flipped  |  {nleft} left, "
              f"~{nleft*est/3600:.1f} h "
              f"({'fits' if nleft*est*SAFETY < DEADLINE-time.time() else 'WILL NOT FIT'})")

    df = pd.DataFrame(rows)
    nf = int(df.now_success.fillna(False).astype(bool).sum()) if len(df) else 0
    # fillna before summing: a resumed row from a 13-column run would make
    # this an object column containing NaN, and int(nan) raises.
    nw = int(df.within_published_budget.fillna(False).astype(bool).sum()) if len(df) else 0
    rate = 100.0 * nf / len(df) if len(df) else None
    summary.append(dict(tag=cfg["tag"], rung=cfg["rung"], n=len(df), n_flipped=nf,
                        n_within_published=nw, rate=rate,
                        n_censored_total=n_cen, n_total=N_EVAL,
                        complete=(len(df) == len(chosen))))
    print(f"  {nf}/{len(df)} flipped"
          + (f" ({rate:.0f}%)" if rate is not None else "")
          + f"   of which {nw} inside the published 400-step restart-0 budget")
    rows_all += rows
    if len(sub_all):
        unload()

print(f"\nSESSION DONE in {(time.time()-SESSION_START)/3600:.2f} h")


## 8 · What it means, with an interval

A flip rate here is a **lower bound** on the false-negative rate: the budget is
capped, so an utterance that did not flip is simply not shown to be a false
negative. Wilson intervals, because at 16 trials a normal approximation on a
proportion near 1 is wrong in the direction that would flatter the result.


In [ ]:
sm = pd.DataFrame(summary)
print(sm.to_string(index=False))


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    m = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5)
    return ((c - m) / d, (c + m) / d)


print("\n--- lower bound on the false-negative rate, by exposure " + "-" * 18)
got = [r for r in summary if r.get("n")]
for r in sorted(got, key=lambda r: r["rung"]):
    lo, hi = wilson(r["n_flipped"], r["n"])
    print(f"  {r['tag']:18s} rung {r['rung']:>3}   "
          f"{r['n_flipped']:2d}/{r['n']:2d} = {r['rate']:5.1f}%   "
          f"Wilson 95% [{100*lo:4.1f}%, {100*hi:5.1f}%]"
          + ("" if r.get("complete") else "   PARTIAL"))
    print(f"                     {r['n_within_published']} of those flipped inside "
          f"the published 400-step restart-0 budget, i.e. pure nondeterminism")

if len(got) >= 2:
    a, b = min(got, key=lambda r: r["rung"]), max(got, key=lambda r: r["rung"])
    la, ha = wilson(a["n_flipped"], a["n"])
    lb, hb = wilson(b["n_flipped"], b["n"])
    print(f"\n  low ({a['tag']}) against high ({b['tag']}):")
    print(f"    {a['rate']:.1f}% [{100*la:.1f}, {100*ha:.1f}] vs "
          f"{b['rate']:.1f}% [{100*lb:.1f}, {100*hb:.1f}]")
    overlap = not (ha < lb or hb < la)
    if overlap:
        print("    Intervals OVERLAP. This sample cannot show the rate differs")
        print("    with exposure, so the censoring bias is consistent with being")
        print("    uniform -- which ATTENUATES the slope rather than confounding")
        print("    it. Report the slope as biased upward, not as an artefact.")
    else:
        print("    Intervals are DISJOINT. The false-negative rate differs with")
        print("    exposure, so the censoring bias is DIFFERENTIAL and every")
        print("    cross-condition eps50 comparison is confounded, not merely")
        print("    attenuated. Feed both rates into reanalysis.py section 6c-3.")

print("\n--- how far past the published budget a flip needed to go " + "-" * 15)
if rows_all:
    da = pd.DataFrame(rows_all)
    fl = da[da.now_success]
    if len(fl):
        print(fl.groupby(["tag", "restart_used"]).steps_used
                .describe()[["count", "50%", "max"]].to_string())
        print("\n  restart_used 0 means it flipped from the same zero start the")
        print("  published attack used; 1 means it needed a random restart.")
else:
    print("  nothing measured.")

import json as _json
prov = dict(PROV)
prov.update(dict(notebook=NOTEBOOK, edit_type=EDIT_TYPE, eps_hi=EPS_HI,
                 n_restarts=N_RESTARTS, strong_steps=STRONG_STEPS,
                 lr_scale=LR_SCALE, use_amp=USE_AMP, check_every=CHECK_EVERY,
                 sample_n=SAMPLE_N, sample_seed=SAMPLE_SEED,
                 summary=summary,
                 session_hours=round((time.time()-SESSION_START)/3600, 2)))
with open(f"{WORK}/_provenance.json", "w") as fh:
    _json.dump(prov, fh, indent=2)

with open(f"{WORK}/dataset-metadata.json", "w") as fh:
    _json.dump({"title": "false-negative-rate-capped",
                "id": "YOURUSERNAME/false-negative-rate-capped",
                "licenses": [{"name": "CC0-1.0"}]}, fh, indent=2)

print("\nfiles in /kaggle/working:")
for f in sorted(os.listdir(WORK)):
    print("   ", f)
print("\nCHECK THIS FILE LIST before publishing: publishing REPLACES the dataset")
print("and deletes anything this run did not regenerate.")
if any(not r.get("complete", True) for r in summary if r.get("n")):
    print("\n*** NOT EVERYTHING FINISHED. Publish, attach the output back as an")
    print("    input, and re-commit: finished rows are skipped, not redone. ***")
